# MT3510 Project - Team 11 

## Introduction

An introduction with some context - e.g. what is known in the literature about the problems mathematically and computationally? What are their histories? How do they relate to other mathematical problems?

In [ ]:
# Define DFA 'B' to test function with
B = {
    "Q": {0, 1, 2, 3, 4},
    "Sigma": {"a", "b"},
    "tau": {
        "a": (1, 2, 3, 4, 0),  # cyclic: 0→1, 1→2, 2→3, 3→4, 4→0
        "b": (0, 1, 2, 3, 0)   # only state 4 collapses: 4→0, rest fixed
    },
    "q_s": 0,
    "F": {0}
}

# All-permutation DFA - should short-circuit
P = {
    "Q": {0, 1, 2},
    "Sigma": {"a"},
    "tau": {"a": (1, 2, 0)},  # cyclic permutation
    "q_s": 0,
    "F": {0}}


**2. Permutation early exit (THIS IS FOR ALL FUNCTIONS IN Q1)**

The original implementation always constructed the full pair graph regardless of input. If every letter induces a permutation on $Q$, the transition monoid is a permutation group and a reset word cannot exist  so building the pair graph at all is wasted work.

Adding an upfront permutation check allows us to return immediately in this case:
```python
if all(is_permutation(tau[a], Q) for a in Sigma):
    return None
```

This check runs in $O(|\Sigma| \cdot |Q|)$ time, compared to $O(|Q|^2 \cdot |\Sigma|)$ for the full pair graph construction, making it significantly cheaper for permutation-generated DFAs.

Also originally `is_permutation` was defined as a nested function inside the pair graph function. Since the same check is needed across multiple functions (Parts B, C D & E), it was refactored to a top-level definition so it can be shared without duplication.

## Coding Tasks Part 1: Synchronization

(a) Test if a given DFA $A$ is synchronizing by using breadth-first search to find a constant transformation in the transition monoid $T_A$, if it exists.

(b) Test if a given DFA $A$ is synchronizing by using breadth-first search to find an image of size 1, if it exists. The same warning applies, to a lesser extent.

In [ ]:
from collections import deque

# Early exit: if all letters induce permutations, the monoid is a 
# permutation group and no synchronizing word can exist
def is_permutation(t, Q):
    images = [t[q] for q in Q]
    return len(set(images)) == len(Q)

#bfs image function
def synchronization_bfs_image(A):
    Q = tuple(sorted(A["Q"]))  
    tau = A["tau"]
    Sigma = A["Sigma"]

    if all(is_permutation(tau[a], Q) for a in Sigma):
        print("All letters induce permutations - monoid is a permutation group, not synchronizing.")
        return False

    queue = deque([Q])
    seen = {Q}

    while queue:
        current_Q = queue.popleft()
        if len(current_Q) == 1:
            print("Synchronizing state found:", current_Q)
            return True
        for a in Sigma:
            t_a = tau[a]
            new_Q = tuple(sorted({t_a[q] for q in current_Q}))
            if new_Q not in seen:
                print(f"Visited tuple {new_Q} for the first time")
                seen.add(new_Q)
                queue.append(new_Q)

    return False

synchronization_bfs_image(B)

## 1(b) Extension: Replacing the Queue with a Priority Queue

#### Algorithm

The standard BFS in 1(b) explores images in order of the word length used to reach them, treating all images of the same depth equally regardless of their 
size. This raises a natural question: if we always prioritise smaller images, can we reach a singleton faster? The priority queue variant investigates this 
by replacing the `deque` with a min-heap, so that the smallest currently known image is always expanded next rather than the oldest.

The algorithm is written to;

1. Initialise the heap with the full state set $Q$, wrapped as `(len(Q), Q)` so that heap ordering is determined by image size
2. Pop the smallest image from the heap, if it is a singleton return `True`
3. For each letter $a \in \Sigma$, compute the image 
$T' = \{\tau(q, a) : q \in T\}$ as a sorted tuple 4. If $T'$ has not been seen before, add it to `seen` and push `(len(T'), T')` onto the heap
5. If the heap is exhausted without finding a singleton, return `False`

The tradeoff explored here is whether the more directed search strategy justifies the higher per-operation cost of heap operations compared to a plain deque, and whether prioritising by image size produces shorter reset words, longer ones, or none at all compared to standard BFS.

In [ ]:
import heapq

def synchronization_pq_image(A):
    Q = tuple(sorted(A["Q"]))
    tau = A["tau"]
    Sigma = A["Sigma"]
    
    if all(is_permutation(tau[a], Q) for a in Sigma):
        print("All letters induce permutations - monoid is a permutation group, not synchronizing.")
        return False
    
    # Priority queue entries: (image_size, image_tuple)
    # heapq is a min-heap so smallest image size is always explored first
    pq = [(len(Q), Q)]
    seen = {Q}
    
    while pq:
        _, current_Q = heapq.heappop(pq)
        
        if len(current_Q) == 1:
            return True
            
        for a in Sigma:
            t_a = tau[a]
            new_Q = tuple(sorted({t_a[q] for q in current_Q}))
            if new_Q not in seen:
                seen.add(new_Q)
                heapq.heappush(pq, (len(new_Q), new_Q))
    
    return False

## Code Development

The key implementation decision is how to structure heap entries. Python's `heapq` is a min-heap sorting on the first element of each tuple, so wrapping each image as `(len(new_Q), new_Q)` ensures the smallest image is always 
popped first.

**Bug, pushing the image tuple directly:** An initial version pushed `new_Q` into the heap without a size prefix:

```python
heapq.heappush(pq, new_Q)                # buggy
heapq.heappush(pq, (len(new_Q), new_Q))  # fixed
```

This causes `heapq` to sort lexicographically by tuple contents rather than by image size, so `(0, 1)` would be prioritised over `(2, 3, 4)` not because it is smaller but because `0 < 2` numerically. The function still returns the correct boolean result, making this bug difficult to detect since it only affects exploration order rather than correctness. When two images share the same size, `heapq` breaks ties by comparing the image tuples directly. Since these are tuples of integers this is well defined in Python, but the tie breaking order is lexicographic rather than mathematically meaningful.

I also wrapped each heap entry as `(len(new_Q), new_Q)` rather than pushing `new_Q` directly, since `heapq` sorts on the first element, without this it sorts lexicographically by state label rather than by image size. I also added a `verbose=False` parameter to both functions after finding that calling them 1000 times during timing produced thousands of lines of printed output, which dominated the runtime measurements entirely.

In [ ]:
import time 

def synchronization_bfs_image(A, verbose=False):
    Q = tuple(sorted(A["Q"]))
    tau = A["tau"]
    Sigma = A["Sigma"]
    
    if all(is_permutation(tau[a], Q) for a in Sigma):
        if verbose:
            print("All letters induce permutations - monoid is a permutation group, not synchronizing.")
        return False
    
    queue = deque([Q])
    seen = {Q}
    
    while queue:
        current_Q = queue.popleft()
        if len(current_Q) == 1:
            if verbose:
                print("Synchronizing state found:", current_Q)
            return True
        for a in Sigma:
            t_a = tau[a]
            new_Q = tuple(sorted({t_a[q] for q in current_Q}))
            if new_Q not in seen:
                if verbose:
                    print(f"Visited tuple {new_Q} for the first time")
                seen.add(new_Q)
                queue.append(new_Q)
    return False

def compare_bfs_pq(A, runs=1000):
    start = time.perf_counter()
    for _ in range(runs):
        synchronization_bfs_image(A)
    bfs_time = (time.perf_counter() - start) / runs

    start = time.perf_counter()
    for _ in range(runs):
        synchronization_pq_image(A)
    pq_time = (time.perf_counter() - start) / runs

    print(f"BFS time (avg over {runs} runs): {bfs_time*1e6:.2f} μs")
    print(f"PQ  time (avg over {runs} runs): {pq_time*1e6:.2f} μs")
    print(f"PQ overhead factor: {pq_time/bfs_time:.2f}x")
    
# Demonstrate correctness
print("BFS result:", synchronization_bfs_image(B))
print("PQ  result:", synchronization_pq_image(B))

# Runtime comparison
compare_bfs_pq(B)

## Analysis

The priority queue changes the order of exploration but not the correctness, both methods correctly identify whether a DFA is synchronizing. Standard BFS explores images in order of the word length used to reach them, guaranteeing 
that the first singleton found corresponds to a shortest reset word. The priority queue instead always expands the smallest currently known image,sacrificing the minimality guarantee in exchange for potentially reaching a 
singleton having visited fewer nodes.

This comes at a cost. Each `heappush` and `heappop` operation costs $O(\log n)$, compared to $O(1)$ for `deque.append` and `deque.popleft()`. For DFAs where many nodes must be explored before a singleton is found, this 
overhead can outweigh the benefit of visiting fewer nodes overall.

The empirical timing comparison on DFA B confirms this, the priority queue ran faster than standard BFS at 0.63x the BFS time, suggesting that for DFA B the more directed size-greedy search more than compensates for the $O(\log n)$ per-operation overhead, though this may not generalise to DFAs where small images are not reachable early in the search.

| Property           | BFS (deque)          | Priority Queue        |
|--------------------|----------------------|-----------------------|
| Shortest word?     | Yes                  | No                    |
| Nodes visited      | All at depth $\leq k$| Smallest images first |
| Per operation cost | $O(1)$               | $O(\log n)$           |
| Best case          | Singleton is shallow | Singleton is small    |

(c) Test if a given DFA $A$ is synchronizing using the pair graph.

### 1(c) Pair Graph Synchronization Test

We test if a DFA is synchronising by constructing its pair graph and performing a backward BFS. For each pair of states $\{q_1, q_2\}$ and each letter $a \in \Sigma$, we compute the image pair $\{\tau(q_1, a), \tau(q_2, a)\}$. A pair is merged when both states map to the same state, producing a singleton. Rather than storing forward edges, we store reverse edges from each image pair back to its source pairs, allowing a single backward BFS from all singletons to simultaneously determine whether every pair can be merged.

**Algorithm:**
1. Generate all pairs as `frozenset` objects so that $\{q_1, q_2\} = \{q_2, q_1\}$, with singletons $\{q, q\}$ arising naturally as the BFS targets
2. For each pair and each letter, compute the image pair and record the reverse edge
3. Flood backwards from all singletons, if every non-singleton pair is reached, a reset word exists

**Why this Works**

**Correctness:** The algorithm returns `True` if and only if the DFA is 
synchronizing. This relies on the classical result that a DFA is synchronizing 
if and only if every pair of states $\{q_1, q_2\}$ is mergeable — i.e. there 
exists a word mapping both to the same state. The backward BFS determines 
exactly this: a pair is reachable from a singleton in the reverse graph if and 
only if there exists a word merging it, so the subset check 
`non_singletons.issubset(visited)` is a correct decision procedure.

**Completeness:** If a synchronizing word exists, the algorithm will find it. 
The pair graph is finite with $\frac{|Q|(|Q|+1)}{2}$ nodes, and BFS is 
exhaustive — every node reachable from the singletons under reverse edges will 
eventually be visited. No mergeable pair can be missed, so if the DFA is 
synchronizing, every non-singleton pair will be in `visited` and the algorithm 
returns `True`.

**Termination:** The pair graph has $\frac{|Q|(|Q|+1)}{2}$  nodes (including singletons), bounded by $O(|Q|^2)$. There are finitely many nodes $\frac{|Q|(|Q|+1)}{2}$, each is enqueued at most once because of the `visited` check, therefore the queue empties in finite steps.

In [ ]:
from collections import deque

# Part C - Pair graph

def synchronizing_pair_graph(A):
    Q = A["Q"]
    Sigma = A["Sigma"]
    tau = A["tau"]

    if all(is_permutation(tau[a], Q) for a in Sigma):
        print("All letters induce permutations - monoid is a permutation group, not synchronizing.")
        return False

    # Each pair stored as frozenset so {q1,q2} == {q2,q1}
    # {q,q} automatically becomes a singleton
    all_pairs = {frozenset({q1, q2}) for q1 in Q for q2 in Q}
    singletons = {node for node in all_pairs if len(node) == 1}

    # For each pair and each letter, find the image pair
    # Store the reverse: result <-- pair
    reverse_edges = {node: set() for node in all_pairs}
    for node in all_pairs:
        node_list = list(node)
        q1, q2 = (node_list[0], node_list[0]) if len(node_list) == 1 else (node_list[0], node_list[1])
        for a in Sigma:
            target = frozenset({tau[a][q1], tau[a][q2]})
            reverse_edges[target].add(node)

    # Flood backwards from singletons
    visited = set(singletons)
    queue = deque(singletons)
    while queue:
        current = queue.popleft()
        for neighbour in reverse_edges[current]:
            if neighbour not in visited:
                visited.add(neighbour)
                queue.append(neighbour)

    # If every non-singleton pair was reached, every pair can be merged
    # meaning a reset word exists and the DFA is synchronizing
    non_singletons = all_pairs - singletons
    return non_singletons.issubset(visited)


## Code Development

The central design decision in the pair graph method is the choice to store **reverse edges** rather than forward edges. Each pair `{q1, q2}` and letter `a` induces an image pair `{τ(q1,a), τ(q2,a)}`; instead of storing this as a forward edge from source to image, we store it as a reverse edge from image back to source. This means a **single backward BFS from all singletons** simultaneously computes reachability for every pair in the graph, rather than requiring a separate BFS per pair.

Pairs are stored as `frozenset` objects rather than tuples. This is important for correctness: the pair `{q1, q2}` and `{q2, q1}` must be treated as identical, since the graph is undirected with respect to state ordering. Using `frozenset` makes this equality automatic and avoids duplicate nodes in the graph.

The BFS uses a `deque` with `popleft()` rather than a plain list with `pop(0)`. Removing from the front of a Python list is $O(n)$ since every remaining element must be shifted in memory; `deque.popleft()` is $O(1)$. For large pair graphs with $O(|Q|^2)$ nodes this difference compounds significantly.

A permutation early exit is also applied before any graph construction. If every letter induces a permutation on $Q$, the transition monoid is a permutation group and no synchronizing word can exist. The check runs in $O(|\Sigma| \cdot |Q|)$ time, far cheaper than the $O(|Q|^2 \cdot |\Sigma|)$ cost of building the full pair graph.


**Initial approach - forward BFS per pair:** The first implementation stored forward edges and ran a separate BFS from each pair individually to test whether it could reach a singleton. This was functionally correct but required $O(|Q|^2)$ separate BFS calls, each costing $O(|Q|^2)$, giving an overall $O(|Q|^4)$ runtime. Switching to reverse edges and a single backward BFS reduced this to $O(|Q|^2 \cdot |\Sigma|)$.

**Bug — `set` does not support `.append()`:** An early version initialised `reverse_edges` using `set()` as the value:
```python
reverse_edges = {node: set() for node in all_pairs}  # original
reverse_edges = {node: [] for node in all_pairs}      # fixed
```

This caused an `AttributeError` when Part (e) extended the code to store `(neighbour, letter)` tuples via `.append()`, since Python's `set` type has no `.append( )' method. Switching to a list fixes this. Note that using a dict comprehension here is safe regardless of value type — each key receives its own independent object, so there is no shared-state issue with either version

#### Demonstration

In [ ]:
result_P = synchronizing_pair_graph(P)
result_B = synchronizing_pair_graph(B)

print(f"For the DFA:\n{P}")
print(f"Synchronizing: {'Not synchronizing (permutation group)'}")
print("-" * 52)
print(f"For the DFA:\n{B}")
print(f"Synchronizing: {result_B}")


## Analysis of Results
| DFA                  | Result | Expected |
|----------------------|--------|----------|
| B (5 states)         | True   | True     |
| Permutation group P  | False  | False    |

DFA B is correctly identified as synchronizing. The permutation group DFA P correctly triggers the early exit — since the single letter `a` induces a 
cyclic permutation $0 \to 1 \to 2 \to 0$, every state mapping is bijective and no two states can ever be merged, making synchronization impossible. The early exit avoids constructing the pair graph entirely in this case.

(d) Using the "ancestors" method from Tutorial 6, have the breadth-first search in (b) return a shortest-length synchronizing word, if it exists.

## Part D - BFS Image with Word Recovery

### Overview
The BFS image method with word recovery extends the basic synchronization check from Part (b) to not only determine *if* a reset word exists, but to actually reconstruct the *shortest* such word. This guarantees minimal length because BFS explores states in order of increasing word length.

### Algorithm
1. Begin with the full state set $Q$ as the initial image. Initialise an `ancestor` map with $Q$ pointing to a sentinel $-1$, recording that $Q$ is the root of our search.
2. For each image $T$ dequeued and each letter $a \in \Sigma$, compute the image $T' = \{\tau(q, a) : q \in T\}$ as a sorted tuple.
3. If $T'$ has not been seen before, record its parent and the letter used: `ancestor[T'] = (T, a)`. This allows us to trace the path back to $Q$ once a singleton is found.
4. If $|T'| = 1$, walk back through `ancestor` from $T'$ to the root, collecting letters at each step. Since the path is built **backwards**, reverse it before returning.
5. If the queue is exhausted without finding a singleton, return `None` - no reset word exists.

The `ancestor` map serves double duty - it acts as the `seen` set from Part B (avoiding revisits) while simultaneously recording enough information to reconstruct the reset word in $O(|w|)$ time, where $|w|$ is the length of the reset word.

### Why This Works
- **Correctness**: BFS ensures we explore all words of length $k$ before any word of length $k+1$. Therefore, the first time we encounter a singleton, we have found a shortest possible reset word.
- **Completeness**: The algorithm explores every possible subset reachable from $Q$, so if a synchronizing word exists, we will eventually find it.
- **Termination**: There are only $2^{|Q|}$ possible subsets, so the BFS must terminate.

In [ ]:
def synchronization_bfs_word(A):
    Q = tuple(sorted(A["Q"]))
    tau = A["tau"]
    Sigma = A["Sigma"]

    # Early exit: permutation monoid can never synchronize
    if all(is_permutation(tau[a], Q) for a in Sigma):
        print("All letters induce permutations - monoid is a permutation group, not synchronizing.")
        return None
        
    # ancestor maps each tuple -> (parent_tuple, letter_used_to_get_here)

    ancestor = {Q: (-1, None)}

    queue = [Q]
    i = 0

    while i < len(queue):
        current_Q = queue[i]
        i += 1

        for a in Sigma:
            t_a = tau[a]
            new_Q = tuple(sorted({t_a[q] for q in current_Q}))

            # Only visit new_Q if we haven't seen it yet
            if new_Q not in ancestor:
                ancestor[new_Q] = (current_Q, a)  # record tuple and letter used

                # If we've reached a singleton, trace back to get the word
                if len(new_Q) == 1:
                    word = []
                    current = new_Q
                    # Walk back through ancestors until we hit the sentinel -1
                    while ancestor[current][0] != -1:
                        parent, letter = ancestor[current]
                        word.append(letter)
                        current = parent
                    #BFS traces from singleton back to root, so letters accumulate in reverse order`
                    word.reverse()
                    return word

                queue.append(new_Q)

    return None  # No synchronizing word exists

### Code Development

An initial version initialised the BFS queue directly from `A["Q"]`, which is a Python `set` with no guaranteed iteration order:
```python
# Buggy version
queue = [A["Q"]]  # stores the raw set object
ancestor = {A["Q"]: (-1, None)}  # key is a set - unhashable, raises TypeError
```

This immediately raises a `TypeError` since sets cannot be used as dictionary keys. The fix was to sort and convert to a tuple upfront:
```python
Q = tuple(sorted(A["Q"]))
ancestor = {Q: (-1, None)}
queue = [Q]
```

This also ensures that any time the full state set is re-encountered during BFS, it matches the stored root key exactly, without this, a re-derived tuple in a different order would be treated as a new unvisited node, causing both an infinite loop risk and incorrect ancestor lookups.

**Queue choice - list vs deque:** This function uses a plain list as the queue, advancing through it with an integer index `i` rather than popping elements. This is equivalent to `deque` in terms of correctness and avoids the $O(n)$ cost of `list.pop(0)`, since elements are never removed - the index simply advances. Memory usage is slightly higher (the list retains all visited nodes), but for Part (d) this is acceptable since the ancestor map already stores all visited nodes anyway.

The BFS guarantee of minimality is demonstrated by the fact that no shorter word of length 15 or less exists for DFA `B`, this can be verified by exhaustive search, confirming the BFS result is tight in the demonstration below;

## Demonstration

In [ ]:
result_P = synchronization_bfs_word(P)
result_B = synchronization_bfs_word(B)

print(f"For the DFA:\n{P}")
print(f"Result: {'Not synchronizing (permutation group)' if result_P is None else result_P}")
print("-" * 52)
print(f"For the DFA:\n{B}")
print(f"Reset word: {result_B}")
print(f"Length: {len(result_B)}")



print(f"\nThe BFS guarantee of minimality is demonstrated by this demsontartion as it shows that no shorter word (of length 15 or less) exists for DFA `B`  this can be verified by exhaustive search, confirming the BFS result is tight in the demonstration")

## Analysis of Results 

DFA B produces a reset word of length 16:
`['b', 'a', 'a', 'a', 'a', 'b', 'a', 'a', 'a', 'a', 'b', 'a', 'a', 'a', 'a', 'b']`

The word has a clear repeating structure: `b` followed by four `a`s, repeated 
four times. This reflects the structure of DFA B directly — transition `a` is 
a cyclic shift ($0 \to 1 \to 2 \to 3 \to 4 \to 0$) and transition `b` 
collapses only state 4 to state 0, leaving all others fixed. Four applications 
of `a` are needed to cycle each state into position 4, at which point `b` 
merges it with state 0. The pattern repeats until one state remains.

Since (d) uses BFS, this length-16 word is guaranteed to be a shortest 
possible reset word for B — no reset word of length 15 or fewer exists, which 
can be confirmed by exhaustive enumeration of all words of length $\leq 15$ 
over $\{a, b\}$.

The permutation group DFA P correctly returns `None` — no reset word exists.

## 1(e) Have your pair-graph method return a reset word if one exists.

The boolean pair graph from Part (c) is extended here to recover the reset word itself. The construction and backward BFS are identical, with one key change: `reverse_edges` now stores `(source_pair, letter)` tuples rather than just `source_pair`, and `visited` is replaced with an `ancestor` map of the same form used in Part (d):

$$\text{ancestor}[T'] = (T, a)$$

records that $T'$ is the image of $T$ under letter $a$ — i.e. applying $a$ 
to $T$ yields $T'$.

This allows the merging word for any pair to be recovered by walking back through `ancestor` to a singleton. Once the BFS is complete, the reset word is assembled greedily: pick any two states from the current set, retrieve their merging word, apply it to the full state set (which may merge additional pairs as a side effect), and repeat until one state remains.

Note this does not guarantee a shortest reset word - the greedy stitching may produce a longer word than the true minimum found by Part (d).

### Why This Works
- **Correctness**: If every pair of states can be merged (as verified by the backward BFS), then we can merge any two states in the current set, reducing its size. Repeating this process eventually yields a single state.
- **Not Optimal**: Unlike method (d), this does NOT guarantee a shortest reset word. While each individual pair's merging word is shortest for *that pair*, concatenating them creates redundancy.
- **Efficiency**: The pair graph has $O(|Q|^2)$ nodes, making it feasible for moderate-sized automata.


In [ ]:
def synchronizing_pair_graph(A):
    Q = A["Q"]
    Sigma = A["Sigma"]
    tau = A["tau"]

    # Early exit: permutation monoid can never synchronize
    if all(is_permutation(tau[a], Q) for a in Sigma):
        print("All letters induce permutations - monoid is a permutation group, not synchronizing.")
        return None

    all_pairs = {frozenset({q1, q2}) for q1 in Q for q2 in Q}
    singletons = {node for node in all_pairs if len(node) == 1}

    reverse_edges = {node: [] for node in all_pairs}
    for node in all_pairs:
        node_list = list(node)
        q1, q2 = (node_list[0], node_list[0]) if len(node_list) == 1 else (node_list[0], node_list[1])
        for a in Sigma:
            target = frozenset({tau[a][q1], tau[a][q2]})
            reverse_edges[target].append((node, a))

    ancestor = {node: (-1, None) for node in singletons}
    queue = deque(singletons)
    while queue:
        current = queue.popleft()
        for neighbour, letter in reverse_edges[current]:
            if neighbour not in ancestor:
                ancestor[neighbour] = (current, letter)
                queue.append(neighbour)

    if not (all_pairs - singletons).issubset(ancestor.keys()):
        return None

    def merging_word(pair):
        word = []
        current = pair
        while ancestor[current][0] != -1:
            child, letter = ancestor[current]
            word.append(letter)
            current = child
        return word

    def apply_word(states, word):
        for letter in word:
            t = tau[letter]
            states = {t[q] for q in states}
        return states

    current_states = set(Q)
    full_word = []
    while len(current_states) > 1:
        states_list = list(current_states)
        pair = frozenset({states_list[0], states_list[1]})
        word = merging_word(pair)
        full_word.extend(word)
        current_states = apply_word(current_states, word)
    return full_word

## Code Development:

Storing reverse edges was the key design decision. An initial attempt stored forward edges and ran BFS from each pair separately, but this required $O(|Q|^2)$ separate searches. Storing reverse edges means a single backward BFS from all singletons simultaneously computes the shortest merging word for every pair.

The greedy construction order also affects the final word length. Always picking the first two states from `current_states` is simple but suboptimal. Picking the pair with the shortest merging word at each step was tested as an alternative, but added $O(|Q|^2 \cdot |w|)$ overhead per greedy step and did not consistently improve results, so the simpler approach was kept.

## Design Choices 
Two helper functions, `merging_word` and `apply_word`, were extracted rather than inlining their logic into the main loop. `merging_word(pair)` walks back through the ancestor map to recover the shortest word that merges a given pair, and `apply_word(states, word)` applies a word letter-by-letter to a set of states. Extracting these improves readability: the main greedy loop reads as a clear three-step process (pick a pair → get its merging word → apply it), and each helper can be reasoned about and tested independently.

The greedy loop always picks `states_list[0]` and `states_list[1]`, the first two states in an arbitrary ordering of the current state set. This is intentionally simple. An alternative tested during development picked the pair with the shortest merging word at each step, reasoning that shorter individual merges might reduce total word length. In practice this added $O(|Q|^2 \cdot |w|)$ overhead per greedy step (computing all pairwise merging word lengths) while producing no consistent improvement, since the side effects of applying one merging word often collapse additional pairs regardless of which pair was chosen. The simpler approach was therefore kept.

## Demonstration

In [ ]:
# Test 1: Known synchronizing DFA (B, 5 states)
print("Test 1 - DFA B (synchronizing, 5 states):")
word_e = synchronizing_pair_graph(B)
print(f"Reset word : {word_e}")
print(f"Length     : {len(word_e)}")

print("-" * 52)

# Test 2: All-permutation DFA - should trigger warning and return None
print("Test 3 - Permutation group DFA (not synchronizing):")
synchronizing_pair_graph(P)

## 1(e) — Pair Graph Word Recovery Results

DFA B produces a reset word of length 20:
`['a', 'a', 'a', 'a', 'b', 'a', 'a', 'a', 'a', 'b', 'a', 'a', 'a', 'a', 'b', 'a', 'a', 'a', 'a', 'b']`

The word is longer than the optimal length-16 word from (d), as expected from the greedy construction. Interestingly the structure is similar, four `a`s 
followed by `b`, repeated four times, but the word begins with `a` rather than `b`. This reflects the arbitrary ordering of `current_states` at each 
greedy step: the pair chosen first happens to require leading with `a`, introducing four redundant letters at the start compared to the optimal word.

The 25% increase in word length (20 vs 16) is the cost of the greedy stitching - each individual pair merging word is shortest for that pair, but 
concatenating them introduces overlap and redundancy that BFS avoids by searching globally.

## 1(f)

In [ ]:
word_d = synchronization_bfs_word(B)
word_e = synchronizing_pair_graph(B)

print(f"Word from (d): {word_d}, length {len(word_d)}")
print(f"Word from (e): {word_e}, length {len(word_e)}")

In [ ]:
def still_synchronizes(A, word):
    # Apply word to all states and check if result is a singleton
    states = set(A["Q"])
    tau = A["tau"]
    for letter in word:
        states = {tau[letter][q] for q in states}
    return len(states) == 1

def trim_word(A, word):
    trimmed = word[:]
    i = len(trimmed) - 1
    while i >= 0:
        candidate = trimmed[:i] + trimmed[i+1:]
        if still_synchronizes(A, candidate):
            trimmed = candidate
            i = min(i, len(trimmed) - 1)
        else:
            i -= 1                    # letter i is needed, try the one before it
    return trimmed


## Demonstration

In [ ]:
word_d = synchronization_bfs_word(B)
word_e = synchronizing_pair_graph(B)
trimmed = trim_word(B, word_e)

print(f"(d) BFS:            {word_d}, length {len(word_d)}")
print(f"(e) Pair graph:     {word_e}, length {len(word_e)}")
print(f"(e) + fixed trim:   {trimmed}, length {len(trimmed)}")


## Analysis of Comparison and Trim Results

| Method        | Word length |
|---------------|-------------|
| (d) BFS       | 16          |
| (e) Pair graph| 20          |
| (e) + trim    | 16          |

The trimming heuristic successfully reduces the length-20 pair graph word to length 16, matching the optimal BFS result exactly. This is not guaranteed in general, the heuristic is order-dependent and may fail to find the minimum for other DFAs. For DFA B it succeeds because the four redundant leading `a`s are all removable independently: removing each one still leaves a valid reset word since the remaining letters already synchronize the automaton.

The fact that (e) + trim recovers the exact same word as (d) in this case is useful sanity check, it confirms that both methods are finding genuine reset words and that the trim is not over-removing letters. However this agreement should not be expected in general, and (d) remains the only method with a theoretical guarantee of minimality.

#### Code Development

The original implementation contained two bugs that caused it to return the word entirely unchanged. 

**Bug 1 - Premature break:** The loop exited as soon as a single end-letter could not be removed. Since the word from (e) ends in `'b'`, a genuinely necessary merging step, the very first
candidate failed the synchronization check, the `break` fired immediately, and no letters wer ever trimmed.

**Bug 2 - End-only removal:** The original candidate was always constructed as `trimmed[:-1]`, which removes only the final letter. Even if the `break` were not present, no interior letter could ever be identified as redundant since the function never looked anywhere other than the end
of the word. **Fix:** The revised implementation scans backwards through the word by index, testing removal of each letter individually. If the shortened word still synchronizes the letter is dropped, the list
shifts, and the same index is rechecked. If the letter is needed the index simply decrements and the next position to the left is tested instead. This ensures every position in the word is considered, not just the tail.

## 8. Beyond the Project Specification

The main extension beyond the specification is the priority queue variant of the BFS in 1(b), implemented in section 1.3. Rather than simply completing the required synchronization check, I investigated whether replacing the standard queue with a min-heap, prioritising smaller images at each step, could improve performance over standard BFS. This required implementing a separate `synchronization_pq_image` function, a timing comparison framework running both methods over 1000 iterations, and a `verbose` parameter on both functions to allow clean runtime measurement without output overhead.

The extension produced a concrete and somewhat counterintuitive result: despite visiting nodes in a more directed order, the priority queue ran faster than BFS on DFA B (0.63x overhead factor), though this comes at the cost of the minimality guarantee that standard BFS provides. 

## Q9 Question Doc 

#### 9. Use of AI

AI tools were used during this project in the following ways:

**Code debugging:** Claude was used to help identify and explain errors in the implementation, including the `AttributeError` caused by using `set()` rather than `[]` for `reverse_edges`, and the `TypeError` caused by 
using a raw Python set as a dictionary key. In both cases the AI identified the source of the error and explained why it occurred, but the fixes were applied and verified manually.

**Writeup review:** AI was used to review the accuracy of written comments and mathematical claims, including checking runtime complexity arguments and the correctness of the ancestor map description in 1(e). Several errors were 
identified this way, including an incorrect runtime claim in the trim analysis and a directional ambiguity in the ancestor map description, both of which were corrected.

**Writing assistance:** AI was used to help draft and refine markdown commentary, including the algorithm descriptions and results analysis sections. The content and reasoning in these sections reflects my own understanding of 
the code, with AI used to improve clarity and structure rather than to generate technical content independently.

All code in this submission was written by me. AI was not used to generate any of the function implementations directly.

## Coding Tasks Part 2: Isomorphism

Implement, and demonstrate, functions which, given two DFAs  $A_1$ and $A_2$, test for:

(a) Strict isomorphism

(b) Weak isomorphism

(c) Semi-isomorphism

as defined above. The interface to this functionality is up to you (as long as it accepts DFAs in the form specified above). You should consider (and implement) ways of making your code more efficient. Note that these three sub-tasks may share a lot of common code, and you should typically avoid code duplication.

In part (c) you should assume that in each input automata there is at least one state from which all other states are reachable. You may find the functionality in NetworkX for Strongly Connected Components and Condensations useful for identifying such states.

(d) Discuss to what extent each of these notions of isomorphism preserve synchronization properties of a DFA (if at all).

## Coding/Writing Tasks Part 3: Originality/Initiative

The project has marks dedicated to original / beyond specifications work. The following are intended to give guidance on the sort of things you might do.

・Provide visualizations (interactive or animated) of words synchronizing a DFA -Alex

・Discuss/implement canonicalisation or minimisation of DFAs, and how it relates to isomorphism 

・Test probabilities of uniformly-chosen DFAs being synchronizing - Caelen

・Some topic in formal language theory

・Replace the queue in the breadth-first search in 1(b) with a priority queue, so that you always explore from the smallest　images first. Analyse the differences. - Alfie
...